# DR Learner with Confounders

This notebook fits a Doubly Robust (DR) Learner for causal effect estimation, controlling for:

1. **Prompt embeddings** (768-dim): Semantic content of user prompts
2. **Model ID** (one-hot): Which LLM model was used
3. **Time of day** (4 bins): Morning (6AM-12PM), Afternoon (12PM-6PM), Evening (6PM-12AM), Night (12AM-6AM)
4. **User embeddings** (16-dim): Hashed user IDs to capture stable user traits

**Treatment**: High empathy (score ≥ 5) vs Low empathy (score < 5)  
**Outcome**: User attachment score

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from scipy.optimize import linear_sum_assignment
from sklearn.neighbors import NearestNeighbors
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv(
    "wildchat_full_scored_mistral.csv",
    engine='python',
    on_bad_lines='skip'   # skip rows that are clearly trash
)

# NA check first
df = df.dropna(subset=['attachment_score'])
df = df[df['empathy_score'] != 4].copy()

df['treatment_group'] = np.where(df['empathy_score'] >= 5, 1, 0) #is 0 if not 5 in Sempathy
df = df[df['user_prompt'].notna()].copy()
df['user_prompt'] = df['user_prompt'].astype(str).str.strip()
df = df.drop_duplicates(subset=['user_prompt']).copy()

init_df = df.copy()

# Get prompt embeddings
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
prompt_embeddings = model.encode(init_df['user_prompt'].tolist(), normalize_embeddings=True)

print(f"Dataset shape after cleaning: {init_df.shape}")
print(f"Prompt embeddings shape: {prompt_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dataset shape after cleaning: (3034, 17)
Prompt embeddings shape: (3034, 768)


### Step 1 — Fit Propensity Model (Logistic Regression)

### Step 2 — Match Treated to Control Using Nearest Neighbors on PS

## Confounder Engineering

Now we'll add confounders to control for in the DR Learner:
1. **Model ID**: One-hot encode the LLM models
2. **Time of Day**: Bin hours into 4 periods (6AM-12PM, 12PM-6PM, 6PM-12AM, 12AM-6AM)
3. **User ID**: Create user embeddings (16-dim) to capture stable user traits

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import FeatureHasher

# 1. ONE-HOT ENCODE MODEL IDs
print("Creating model ID features...")
model_dummies = pd.get_dummies(init_df['model'], prefix='model')
print(f"Model one-hot shape: {model_dummies.shape}")
print(f"Unique models: {init_df['model'].nunique()}")

# 2. BIN HOUR OF DAY INTO 4 PERIODS
print("\nCreating time-of-day bins...")
def hour_to_period(hour):
    """Convert hour (0-23) to period bin"""
    if pd.isna(hour):
        return 'unknown'
    hour = int(hour)
    if 6 <= hour < 12:
        return 'morning'      # 6AM-12PM
    elif 12 <= hour < 18:
        return 'afternoon'    # 12PM-6PM
    elif 18 <= hour < 24:
        return 'evening'      # 6PM-12AM
    else:
        return 'night'        # 12AM-6AM

init_df['time_period'] = init_df['hour_of_day'].apply(hour_to_period)
time_dummies = pd.get_dummies(init_df['time_period'], prefix='time')
print(f"Time period distribution:\n{init_df['time_period'].value_counts()}")
print(f"Time dummies shape: {time_dummies.shape}")

# 3. CREATE USER ID EMBEDDINGS (16-dimensional)
print("\nCreating user ID embeddings...")
unique_users = init_df['user_id'].unique()
n_users = len(unique_users)
print(f"Unique users: {n_users}")

# Use FeatureHasher to create low-dimensional user embeddings
# This hashes user IDs into a fixed 16-dimensional space
hasher = FeatureHasher(n_features=16, input_type='string')
user_ids_list = [[uid] for uid in init_df['user_id'].values]
user_embeddings = hasher.transform(user_ids_list).toarray()

print(f"User embeddings shape: {user_embeddings.shape}")
print(f"User embeddings range: [{user_embeddings.min():.3f}, {user_embeddings.max():.3f}]")

Creating model ID features...
Model one-hot shape: (3034, 2)
Unique models: 2

Creating time-of-day bins...
Time period distribution:
time_period
afternoon    908
night        777
morning      729
evening      620
Name: count, dtype: int64
Time dummies shape: (3034, 4)

Creating user ID embeddings...
Unique users: 898
User embeddings shape: (3034, 16)
User embeddings range: [-1.000, 1.000]


In [4]:
# COMBINE ALL FEATURES: Prompt embeddings + Confounders
print("\nCombining all features...")

# Concatenate: [prompt_embeddings, model_dummies, time_dummies, user_embeddings]
X = np.hstack([
    prompt_embeddings,           # (n_samples, 768) - semantic content
    model_dummies.values,        # (n_samples, n_models) - which LLM
    time_dummies.values,         # (n_samples, 4 or 5) - time of day
    user_embeddings              # (n_samples, 16) - user traits
])

print(f"Final feature matrix X shape: {X.shape}")
print(f"  - Prompt embeddings: {prompt_embeddings.shape[1]} dims")
print(f"  - Model dummies: {model_dummies.shape[1]} dims")
print(f"  - Time dummies: {time_dummies.shape[1]} dims")
print(f"  - User embeddings: {user_embeddings.shape[1]} dims")
print(f"  - Total: {X.shape[1]} features")


Combining all features...
Final feature matrix X shape: (3034, 790)
  - Prompt embeddings: 768 dims
  - Model dummies: 2 dims
  - Time dummies: 4 dims
  - User embeddings: 16 dims
  - Total: 790 features


# **DR LEARNER**

In [5]:
!pip install econml
from econml.grf import CausalForest
from econml.dr import DRLearner
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Ensure clean index
init_df = init_df.reset_index(drop=True)

y = init_df['attachment_score'].values.astype(float)
t = init_df['treatment_group'].values.astype(int)

print(f"Treatment distribution: {np.bincount(t)}")
print(f"Outcome (attachment) - mean: {y.mean():.3f}, std: {y.std():.3f}")
print(f"Feature matrix shape: {X.shape}")

# Nuisance models - use larger forests for better confounder control
reg = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=0)
clf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0)

dr = DRLearner(
    model_regression=reg,
    model_propensity=clf,
    random_state=0
)

print("\nFitting DR Learner with confounders (model_id, time_of_day, user_embeddings)...")
dr.fit(Y=y, T=t, X=X)
tau_hat = dr.effect(X)

print("\n" + "="*60)
print("CAUSAL EFFECT ESTIMATES")
print("="*60)
print(f"ATE (Average Treatment Effect): {np.mean(tau_hat):.4f}")
print(f"Std of CATEs: {np.std(tau_hat):.4f}")
print(f"Min CATE: {np.min(tau_hat):.4f}")
print(f"Max CATE: {np.max(tau_hat):.4f}")
print(f"Median CATE: {np.median(tau_hat):.4f}")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 26.1 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found existing installation: shap 0.50.0
    Uninstalling shap-0.50.0:
      Successfully uninstalled shap-0.50.0
Treatment distribution: [2568  466]
Outcome (attachment) - mean: 1.887, std: 1.149
Feature matrix shape: (3034, 790)

Fitting DR Learner with confounders (model_id, time_of_day, user_embeddings)...


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/linear_model.py:1815: UserWarning: Co-variance matrix is underdetermined. Inference will be invalid!
  warnings.warn("Co-variance matrix is underdetermined. Inference will be invalid!")



CAUSAL EFFECT ESTIMATES
ATE (Average Treatment Effect): 0.8175
Std of CATEs: 1.6615
Min CATE: -7.2911
Max CATE: 8.8161
Median CATE: 0.8543


## Confounder Analysis

Let's examine how the confounders relate to treatment and outcome:

In [6]:
# Check confounder balance across treatment groups
print("CONFOUNDER BALANCE ACROSS TREATMENT GROUPS")
print("="*60)

# Model distribution
print("\n1. MODEL DISTRIBUTION:")
model_treat = pd.crosstab(init_df['model'], init_df['treatment_group'], normalize='columns')
print(model_treat.round(3))

# Time of day distribution
print("\n2. TIME OF DAY DISTRIBUTION:")
time_treat = pd.crosstab(init_df['time_period'], init_df['treatment_group'], normalize='columns')
print(time_treat.round(3))

# Check if confounders predict treatment (they should if confounding exists)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Use only confounder features (exclude prompt embeddings)
confounder_features = X[:, prompt_embeddings.shape[1]:]
print(f"\n3. CONFOUNDERS' PREDICTIVE POWER FOR TREATMENT:")
print(f"Confounder features shape: {confounder_features.shape}")

prop_check = LogisticRegression(max_iter=1000, random_state=42)
prop_check.fit(confounder_features, t)
t_pred_prob = prop_check.predict_proba(confounder_features)[:, 1]
auc = roc_auc_score(t, t_pred_prob)
print(f"AUC (confounders → treatment): {auc:.4f}")
print("(Higher AUC indicates confounders are important for treatment assignment)")

# Check if confounders predict outcome
print(f"\n4. CONFOUNDERS' PREDICTIVE POWER FOR OUTCOME:")
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

outcome_check = Ridge(alpha=1.0, random_state=42)
outcome_check.fit(confounder_features, y)
y_pred = outcome_check.predict(confounder_features)
r2 = r2_score(y, y_pred)
print(f"R² (confounders → attachment score): {r2:.4f}")
print("(Higher R² indicates confounders are important for the outcome)")

CONFOUNDER BALANCE ACROSS TREATMENT GROUPS

1. MODEL DISTRIBUTION:
treatment_group         0      1
model                           
gpt-3.5-turbo-0301  0.807  0.809
gpt-4-0314          0.193  0.191

2. TIME OF DAY DISTRIBUTION:
treatment_group      0      1
time_period                  
afternoon        0.302  0.283
evening          0.201  0.225
morning          0.236  0.262
night            0.261  0.230

3. CONFOUNDERS' PREDICTIVE POWER FOR TREATMENT:
Confounder features shape: (3034, 22)
AUC (confounders → treatment): 0.6304
(Higher AUC indicates confounders are important for treatment assignment)

4. CONFOUNDERS' PREDICTIVE POWER FOR OUTCOME:
R² (confounders → attachment score): 0.0408
(Higher R² indicates confounders are important for the outcome)
